# Messages and queues

Diluvium is a messaging runtime as well as a language: a program can
hold bounded queues, park on them, and be driven by a host that moves
bytes between instances.

This notebook is the guest half — the part a *program* sees. It needs
**5.5.1_build3 or newer**; the cell below says whether you have it.

In [ ]:
local have = type(queue) == "table" and type(msgpack) == "table"
print("queue:   " .. type(queue))
print("msgpack: " .. type(msgpack))
print("endpoint:" .. type(endpoint))
print(have and "-- you are good to go" or
  "-- this build predates the messaging layer; pick a newer runtime")

## msgpack

A codec that round-trips a Lua value graph. Integers stay integers and
floats stay floats, which JSON cannot promise.

In [ ]:
local encoded = msgpack.encode{
  name = "ada", years = 36, active = true,
  tags = { "maths", "engines" },
}
print($"{#encoded} bytes")

local back = msgpack.decode(encoded)
print(back.name, back.years, back.active, back.tags[2])

In [ ]:
local round = msgpack.decode(msgpack.encode{ i = 3, f = 3.0 })
print(math.type(round.i), math.type(round.f))

### The one trap that will get you

**An empty table encodes as a map, not an array.**

A table is an array if it has at least one element and its keys are
exactly `1..n`; otherwise it is a map. So `{}` is a map — and a list you
happened to empty *changes shape on the wire*.

This has bitten Diluvium itself more than once, including a capability
list refused as malformed because `caps = {}` arrived as a map.

In [ ]:
local function first_byte(v) return ("%02x"):format(msgpack.encode(v):byte(1)) end

print("{}        -> " .. first_byte({}) .. "   (0x80 = map, 0 pairs)")
print("{1,2,3}   -> " .. first_byte({1,2,3}) .. "   (0x93 = array, 3 items)")

If the far side cares, say which you meant.

In [ ]:
local function first_byte(v) return ("%02x"):format(msgpack.encode(v):byte(1)) end

print("as_array({}) -> " .. first_byte(msgpack.as_array({})) .. "   an empty array")
print("as_map({1,2}) -> " .. first_byte(msgpack.as_map({1,2})) .. "   a map keyed 1 and 2")

### Ext codes

`0x10` to `0x7F` are yours. Below `0x10` is reserved by the runtime's
own registry — decimals, endpoint references, prototypes, closures — and
refused in **both** directions, so a program cannot produce bytes only
the runtime is supposed to mean something by.

In [ ]:
print(pcall(msgpack.ext, 0x20, "mine"))
print(select(2, pcall(msgpack.ext, 0x02, "reserved")))

## Queues

Bounded, named, and declared by the program that owns them. Every
instance starts with `inbox` and `outbox` already declared; anything
else you declare yourself.

In [ ]:
local work = queue.declare("work", { capacity = 4 })
print($"declared id {work}, capacity {queue.capacity(work)}, length {queue.len(work)}")

queue.push(work, { task = "build", n = 1 })
queue.push(work, { task = "test",  n = 2 })
print($"after two pushes: {queue.len(work)}")

local m = queue.pop(work)
print($"popped: {m.task} #{m.n}")

`push` **reports** whether the message was accepted rather than
raising. A full queue is an ordinary answer, not an error — what to do
about it is policy, and policy belongs to the program.

In [ ]:
local tiny = queue.declare("tiny", { capacity = 1 })
print(queue.push(tiny, { a = 1 }))
print(queue.push(tiny, { b = 2 }))

`queue.info` reports everything about a queue in one table.

In [ ]:
local q = queue.declare("described", { capacity = 8 })
local info = queue.info(q)
local keys = {}
for k in pairs(info) do keys[#keys + 1] = k end
table.sort(keys)
for _, k in ipairs(keys) do print($"{k}: {tostring(info[k])}") end

### A queue must exist before anything can be pushed to it

It exists once the program that declares it has *run*. So a host cannot
push into an instance it has not driven yet: drive first, then push.
`lookup` on a name nobody declared is nil, not an error.

In [ ]:
print(queue.lookup("never-declared"))

### Parking on a queue needs a host

`queue.wait` blocks the program until a message arrives — which means
yielding, which means something has to resume it. A notebook cell is
not that something, and the runtime says so in a sentence that lists
every case where it applies.

In [ ]:
local q = queue.declare("parked", { capacity = 1 })
print(select(2, pcall(queue.wait, { q })))

Press **Sandbox** on a cell that calls `queue.wait` and the Lab runs it
as a real instance instead — where parking works, and the panel reports
the program as parked and says what it is waiting on.

See the **Sandboxed instances** notebook.

## The shape of a real program

A supervisor is a program, not a type. It declares `system/lifecycle`
and `system/events`, pushes a spawn request, and reads back what
happened. The event records are `{ event = ..., id = ..., detail = ... }`.

Nothing here can spawn — the swarm layer is not in the browser artifact
— but the queue and the record shape are exactly the real ones, so this
is the loop a supervisor runs.

In [ ]:
local evq = queue.declare("system/events", { capacity = 32 })

-- stand in for what the swarm layer would push
local function emit(what, id, detail)
  queue.push(evq, { event = what, id = id, detail = detail })
end
emit("spawned",  2)
emit("spawned",  3)
emit("denied",   3, "capability not held: lifecycle")
emit("exceeded", 2, "instruction budget: 200000")
emit("exited",   2)

-- the loop a supervisor runs: drain what is there, decide, repeat
local drained = {}
while true do
  local m = queue.pop(evq)
  if not m then break end
  drained[#drained + 1] = m
end

print($"drained {#drained} records")
events(drained, { title = "drained from a real system/events queue" })

The eight event kinds are `spawned`, `exited`, `faulted`, `exceeded`,
`hibernated`, `throttled`, `denied` and `status`.

## Idioms worth knowing early

Collected from things that actually went wrong in Diluvium's own
development:

- **Generate a program per unit of work.** A coordinator cannot send a
  handler a setup message, so it writes the parameters into the
  handler's *source* and spawns that. Use `%q` for every value you
  interpolate — that is the difference between code generation and an
  injection hole.
- **Code is data.** Hand a program its source over a queue like any
  other string.
- **Handles are never reused.** A restarted child is a *different*
  instance with a new id. A host that caches the first id keeps pushing
  at a dead handle, silently, forever.
- **A `spawned` event arrives before the child runs an instruction**, so
  a name-to-id mapping built from it is always in place before that
  child's first message. That ordering is the only one you get.
- **Prefer relative timing to absolute steps.** How many steps a
  handshake takes is not a contract.

In [ ]:
-- %q is the whole of the first idiom
local name = [[she said "hi"
then left]]

local source = ("local who = %q\nreturn who"):format(name)
print(source)
print("round-trips exactly:", load(source)() == name)